# ⚡ Boosting
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> While Random Forests train trees independently and average them, Boosting trains trees *sequentially* — each tree learns from the mistakes of the previous one. This iterative error-correction makes Boosting one of the highest-performing ML approaches.

### What you'll learn
- AdaBoost — upweighting misclassified points
- Gradient Boosting — fitting trees to residuals
- XGBoost — the industry-standard implementation
- Key hyperparameters: n_estimators, learning_rate, max_depth
- Comparing Boosting vs Random Forests

### Dataset: California Housing (regression) + Heart disease (classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
!pip install xgboost -q
import xgboost as xgb
print("✓ XGBoost:", xgb.__version__)

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_house = data.frame
X = df_house.drop('MedHouseVal', axis=1)
y = df_house['MedHouseVal']
print(f"California Housing: {df_house.shape}")
df_house.head()

In [ ]:
# Load California Housing
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_house = data.frame
X = df_house.drop('MedHouseVal', axis=1)
y = df_house['MedHouseVal']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"California Housing: {X.shape[0]:,} census blocks")
print(f"Predicting: median house value (in $100,000s)")
print(f"Features: {list(X.columns)}")

## ⚡ Part 1 — How Gradient Boosting Works

**Core idea:** Train trees sequentially. Each tree fits the *residuals* (errors) from all previous trees.

**Step by step:**
1. Initialize with a simple prediction (e.g. mean of Y)
2. Compute residuals: rᵢ = yᵢ - ŷᵢ
3. Fit a small tree to the residuals
4. Update: ŷᵢ ← ŷᵢ + λ × (tree prediction)  (λ = learning rate, shrinks each step)
5. Repeat 2-4 for B iterations

The **learning rate** (shrinkage) λ is critical — small λ requires more trees but generalizes better.

In [ ]:
# Show boosting building up — predictions improve with each tree
from sklearn.ensemble import GradientBoostingRegressor

stages = [1, 5, 25, 100, 300]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

# Use a small 1D slice for visualization
X_viz = X_tr[['MedInc']].copy()
y_viz = y_tr.values

for ax, n in zip(axes, stages):
    gbm = GradientBoostingRegressor(n_estimators=n, learning_rate=0.1,
                                     max_depth=3, random_state=42)
    gbm.fit(X_viz, y_viz)
    
    xi = np.linspace(X_viz['MedInc'].min(), X_viz['MedInc'].max(), 200).reshape(-1,1)
    yi_pred = gbm.predict(xi)
    
    ax.scatter(X_viz['MedInc'], y_viz, alpha=0.1, s=5, color='#888')
    ax.plot(xi, yi_pred, color='#e85d20', lw=2)
    mse = mean_squared_error(y_viz, gbm.predict(X_viz))
    ax.set_title(f'n={n}\nRMSE={np.sqrt(mse):.3f}', fontsize=10)
    ax.set_xlabel('Median Income')
    if n==1: ax.set_ylabel('House Value')

plt.suptitle('Gradient Boosting — Predictions Improve with More Trees', y=1.05, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: sklearn GBM vs XGBoost
from sklearn.metrics import mean_squared_error as mse

models = {
    'GBM (sklearn)': GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                                max_depth=4, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, learning_rate=0.1,
                                  max_depth=4, random_state=42,
                                  eval_metric='rmse', verbosity=0),
}

print(f"{'Model':<20} {'Train RMSE':>12} {'Test RMSE':>12}")
print("-"*45)
for name, model in models.items():
    model.fit(X_tr, y_tr)
    tr_rmse = np.sqrt(mse(y_tr, model.predict(X_tr)))
    te_rmse = np.sqrt(mse(y_te, model.predict(X_te)))
    print(f"{name:<20} {tr_rmse:>12.4f} {te_rmse:>12.4f}")
print("\n📌 XGBoost uses second-order gradients, regularization, and parallel computing")
print("   — often faster and slightly better than sklearn's GBM")

In [ ]:
# Effect of learning rate and n_estimators (the key tradeoff)
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3]
n_estimators   = [10, 50, 100, 200, 500]
results = np.zeros((len(learning_rates), len(n_estimators)))

for i, lr in enumerate(learning_rates):
    for j, n in enumerate(n_estimators):
        gbm = GradientBoostingRegressor(n_estimators=n, learning_rate=lr,
                                         max_depth=3, random_state=42)
        gbm.fit(X_tr, y_tr)
        results[i,j] = np.sqrt(mse(y_te, gbm.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 4))
for i, lr in enumerate(learning_rates):
    ax.plot(n_estimators, results[i], 'o-', lw=1.8, markersize=5,
            label=f'lr={lr}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('Test RMSE')
ax.set_title('Learning Rate × n_estimators Tradeoff\n(small lr requires many trees but generalizes better)')
ax.legend(title='Learning rate', fontsize=9)
plt.tight_layout()
plt.show()
print("\n📌 Small learning rate (0.01) + many trees often outperforms large lr + few trees")
print("   But small lr is slower to train — XGBoost's early stopping solves this")

In [ ]:
# Feature importance (XGBoost)
xgb_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)
xgb_model.fit(X_tr, y_tr)

imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e85d20' if v > imp.quantile(0.6) else '#1e5fa8' for v in imp]
ax.barh(imp.index, imp.values, color=colors, edgecolor='white')
ax.set_title('XGBoost Feature Importance — California Housing')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()
final_rmse = np.sqrt(mse(y_te, xgb_model.predict(X_te)))
print(f"\nFinal XGBoost Test RMSE: {final_rmse:.4f}")
print("📌 MedInc (median income) dominates — wealthy neighborhoods = higher house values")

In [ ]:
# @title 📝 Quiz — Boosting
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the core difference between Random Forest and Gradient Boosting?
# @markdown **Q2:** What does the learning rate control in gradient boosting?
# @markdown **Q3:** Why does a very high learning rate (e.g. 0.9) often perform poorly?
# @markdown **Q4:** What is the advantage of XGBoost over sklearn's GradientBoosting?
# @markdown **Q5:** If your boosting model overfits, name two hyperparameters you would adjust

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_="Boosting"
# @title 📋 Get AI Feedback on Your Quiz
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2:** Click ▶ Run — your answers are formatted as a prompt
# @markdown
# @markdown **Step 3:** Copy the output → click the **✨ Gemini button** in the Colab toolbar (top right) → paste → send
# @markdown
# @markdown **Step 4:** Keep the conversation going — ask Gemini to explain any output,
# @markdown code block, or chart from this notebook for deeper understanding

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    nd = sum(1 for v in answers.values() if str(v).strip())

    sep = "\u2550" * 62

    print(sep)
    print(f"  Student  : @{GITHUB_USERNAME.strip()}" if GITHUB_USERNAME.strip()
          else "  Student  : (set GITHUB_USERNAME above)")
    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answered : {nd}/{len(answers)} questions")
    print(sep)
    print()
    print("  \u2728  COPY THE PROMPT BELOW INTO GEMINI  \u2728")
    print("  (click the Gemini button in the Colab toolbar, top right)")
    print()
    print("\u2500" * 62)
    print()
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  - Say CORRECT, PARTIAL, or INCORRECT")
    print("  - One sentence: what I got right, or exactly what concept I should review")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - One specific thing to study if I struggled")
    print()
    print("After grading, I may paste specific code outputs or charts from this")
    print("notebook and ask follow-up questions — please help me understand them.")
    print()
    print("\u2500" * 62)
    print()
    print("  After pasting, keep the Gemini chat open and ask things like:")
    print("  \"Why did my SHAP waterfall plot show X?\"")
    print("  \"Can you explain what this output means: [paste output]\"")
    print("  \"I got a different result than the notebook — what went wrong?\"")
    print()
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork")
    print(sep)
